# AeroVision Lab — free Colab GPU pilot
1. Choose **Runtime → Change runtime type → GPU** (a free GPU option, if available).
2. Run the cells in order. Approve Google Drive access when prompted.
3. Keep this notebook private: it includes a snapshot of your private project's source.

Training runs on Google's GPU. Do not connect to a local runtime.
This notebook contains the complete pipeline, so no GitHub token is needed.
Checkpoints save to `My Drive/AeroVisionLab/runs/` every epoch. Colab resources
are not guaranteed; if a session ends, rerun the notebook to prepare the same data
and resume from the saved checkpoint. A failure during the current epoch can lose
that epoch. Select another saved epoch checkpoint if `last.pt` is damaged.

First milestone: YOLO11n, 800 training / 200 validation images, 640-pixel input,
batch 2, 20 epochs. Batch stays small to preserve the planned pilot setup.


In [ ]:
import os, sys, pathlib, subprocess, json, csv, zipfile, io, base64, hashlib
from google.colab import drive
import torch
assert torch.cuda.is_available(), 'Select a hosted GPU runtime, then rerun this cell.'
print('Hosted GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
PERSIST = pathlib.Path('/content/drive/MyDrive/AeroVisionLab')
PERSIST.mkdir(parents=True, exist_ok=True)
PROJECT = pathlib.Path('/content/aerovision-lab')
PROJECT.mkdir(exist_ok=True)
os.chdir(PROJECT)
os.environ['YOLO_CONFIG_DIR'] = str(PROJECT / '.state/ultralytics')


Source snapshot SHA-256: `cbf2cd68c92de9631e4c3684845060d327784d064fcaee969d81dd73064a17fe`. The next cell installs this snapshot into the hosted runtime.

In [ ]:
bundle = base64.b64decode('UEsDBBQAAAAIAAAAIQB4tUodYAEAAFECAAAOAAAAcHlwcm9qZWN0LnRvbWxVUctqwzAQvOsrhM6xsB0IaakMPfXSQqD0UIwIsryO1ciSKslJ/fdFdsjjOLuzO7OzdTMq3WZhChEGjjz8jspDwAzXJEAcXbRWh4pttoSjhdsIeQTTYobvGHTu7QeIgiBUO29/QEaOjBggMQV4e1JBWZNp0RB0Ap9AauW0oDlBLQTplYuX6ucgtMZvuy8Mfw68GsDEgDvrsQCvhMYhETLbJB3cQgSZRsn1hMxNsV92VWxNiyJJODAtGKkuF446eqGnqGRgbEvXtCy3ZIWJGQc3Vayg5Wb1sk6VndLanitW5DOavl8/3iu2ScBN0kp7yamkOU1RXSOgdj5J6OxenaMWTrMFN0UIsWKzrh+7rmI5fXrYsOQSOLqF+BgplVo9D0KZFH0yQu8e44Q8igME2inTcnTuwaeP1CR4OcvMA4sNqozaL4YDR6niROyXrBIKt4FklSOtDGQazCH2mOGiyNE/UEsDBBQAAAAIAAAAIQCeDd5ZIQsAAK8XAAAJAAAAUkVBRE1FLm1kfVhtb9w2Ev6+wP6HAfrhGmNXu3aTXBv3ChhOmwaXFyOOUxyK4sylRhLPFEflULve/vrDDPWySXv3JXC0EjmceeZ5nuFXcIWRPjl2FOCN2S0Xy8Vta7xf0+4/aBOUmNAm+RUfO4yuxZAYKMAnxy8jBXwByMnsvOMGDETsIpW9dTuPsDOM3gVcQWowgAt75ORqkxAiMvle1l1Bch5LcKHCiMEimFBCiZ2no2xWSEgfG8fgGIysUmKHocSQwLWdR3nJaIScTExYQhWpBbbRJNsU8I4gReOCC7Vs2/vE0Jg9wg4xQA4XSzhi3uqrr+A2mRrh/MV0GpQDQDL8IG+cF/DJeFfKOeQ5hr2LFCQOjf3VzR0cXJJ0cEMxAbf0gBD7UCwXFwXcROxMxCmD+WV87LyzLoH1hhla03Uu1CvgzrsErQmuQk6SghJMCDSc2TZoH7hYLr4p4KMcEwz86/2b9+fnPOUfTILnT7fQuUf0XMCtnF4rsMcSGFNyoeYVjIm1R9hjFEjwChixXEFjYnmQoGV73bMjFxI0hhuU7Z8W8OPe+F6SQgH2OUESYWmSuZTtMO4REnJal7iHiiJULhgPmL+Td02VMAKjF8yFOqd3wp0UK5ElXywXzwq4plbz+PzpdgXfPd9qbOcX327Bha5PJxjjsR6WQorkBW8TJhhT3xVw3RAxgoGdwAbY/SHVNQkqJ1n3Hmzer5RK5jpEtBSlHEcoce/0DANeESoXOUFNxgtyE/0FmnSRuX2oj0CHAC0a7qMCmwt4K/FIoCZQajB+gfq/MeCjsQlC3+4wapsESmCY+xbLzzF98UK+jrRHebyeUnhoyOPatfKOxER7jD4j8E/t2bM8tRTYcZKqsKWY4ytnFHPfdRFZeWVEWCFbvj5hAYyRIkNAE3Ub2FEfShMd8mqMIdNBL8vKJy3G2oVal/qAXW6vE76KaI33K+gwrnMnXd2soDKeETpil9xeFm+xpXjUtb1JCnnjKdTsStSdpeDt1Y3u8wmjq45yNlJ+aalED13E0ik1ZjAMMMYT7pIoKoqtCRa1ED/uMR7HEugbbc8JdjiDa2QOI705dTAFBQ2bFrWhclbI9rIIlqedoeXOVEABqogIr4hqj3BNPjO8wNObLlEH3PSJoRTY9aHEqOzlyZQr4BPibAxDSxJSIjDQEMuur27uirwa+RI8WeP1+9iHIEBliJhcxPISrt+8zqtthpNCoAOUWJneJ1n0WtdSdn1JhyAhwK+zOP1bgy9cdwy7374OlHBH9MCbv37jSWba9x0GaFLq+MVmY/Vn4SETbVPUmpPCUrsZSE37/+zsJwHiD3DXaQjjTmdnmWMHmjg7+9CH5Fp587oxoVZ+1wfp2MnTVzd3Z2e6cq+0MqRseCsz5oc+l9Wi9wxSsCglUIrvFCNj6V5Gt0cw1iIzHERNu0htl4YGV30cQwVsd1gyGGFxVznheOqjReBgOm4oaW0DwSuXfu53kOhB5Fmq9Xsv5SqWi38iduCEct3eJCzgpUmGUTTX1MiitUfBlyZ9FFvgRNHUeCnHBBOTq4zNmrVczKrBiu4Bb/dvj/l0J6V8Y3YbIdnNPaCxDWBHouQfUIA16MJyMZ034q53vuS5RbjfSayZo4UH1Tr0oXLBcSMJ0faaQyqWi9cJjGfSn/JSmVeH3O36UHqUImm0BfyknXVzN1ZFN8vVXS68q1D+YtibeLyE9FmBpJuEo48ovY/irdD2Ag8X4CgykNNqrKV+skAInfOUBE88aHxYwbfbbW4s2MDFdnuqvLlUqxPtXw3qdrFcSLgX25xaLuB1yvaJs3u6fn/9ft1F1JWFldDVjaiRhIEmeodxaHh1N8uFtLXHJL0uucu/5YAPRsCdMMZeEAs7rCjioJ6qHDB9PdR6oLHJUuTKSs2GJV1qSDV+pCj5YujZU9oYVhBh/n/c8eVrT5SSMl9KprIl+Zz54K0UZwZd7tGe5YRHfayxSs5Oij80QkbYiZO6/flqffHsucJoBh9FV6tJytrcSmtHbly3ESuDIa2WCzXXA6vyiSyBd614l2+22xU83w7+aLvdZo3RTfKxKeZ3oXKPWC4XJul7BbxU908RqE9iqWSF2tPOeBViHLcQkEOiGsWgKABcFKfu9q7sjV8usKpQmUCJP4FjEt0tM6C+2W5Ha5ZpSCwFlgND5N7Bwxyt9FfrMo2NciJncplO5V3xVFjAlfeT6Y/YSpucWlNn6kCcnBVvsly8kQJyv+siaUsPp3ZCdxFNm+VPtxiqWcBPkrK1pVA5sc4Ie8e9uFrxNssFPhpB9uDb6zqiOp86IpbHdavdqE2ec8MyGsgLUxfC1Q20mGKOMRtLAcS96tlaIbben69nF73eXxR/uO5+ZDIa1UMtQ5ZWVv8dce/wMDbb9c2d2Fj01ImrmIyCdvLpkJMTyWD2xnnJsy428CSW6pdDqeuJ3WdRdMnYciGyIJS5w2Cb1sQHBsqWY3Ia6nQcD/4k40NWwRIaJ/riJJxfXCjpwMvFaVwKHeGUTKT3JVneDJqmZnqtZynSY7q/HCg2YtKzuMTLRRfRBU5GB4Sb40eKtjkx+rlnx9EIjE298f6YW94FWUOFr8VkxKSNiX07+/nJqGX/LYeTAWv99GLULB7wQlXlrDN+IHeJ4pTa5ynYH1fjKZYL0EklYYBkYo3DNCkiMA1uufQC1qSpzMwy0AkYG4k5T50COMjsVRJmyZqnlTkEi0DVgHt/BHat8yYCWww6G67h/UhjYr3EgcNMqpK77Cl0Kh95ME+WktZZTsYpOA8SdVCS0HlFeGccAXb0KBtEBGpdmi4ERDFnmOX0qKDp1YKmLtNZzjBVGYc7w87O31kKAwBeSGBHsCaIfxd+kK2MfGEf6iiTTDHPWI4pZOXujpYsJSKv9yiTfAxJUG5uzeNLTPwPpe6ZWyTIbbHdnuuh3729hdd0B9vi78WUjYh1BqcKrIxgeUAwfDIS2UiHMudpLrFcr5yOFOpAH12r6Vhp8SXWCZfT9cVEy3rcX74YJfP4eKJKYodn9AVMB4oPfxrZs2cE/Ty3rZ53GiXh45frmrLkUZz0mGujNxbv3t4WcJXGeTLn8GIFRnzRWn1RHj5NuZdBjcHsxFo8O78Yb0zkXG+GMdEF6/syC+2gE3pPIyQ2DcmrcU6dLKgMOdEErjCuYNcnCQMfh6U0W+tKgijRkniiIl8jaWoY+BhsEym4PwbAqy2WORQOJoou9Z1W8hZ/7yW1whuy3Dy2V2ZPfWTwdBinX5nh+7bLcN9jhA7NA6QmUl83XZ/bTMbnq5tn29Xnc/bVzeavRm0NrsXSmbD57llq1h1GKwF5mTg1g9ppIthRmjNkwJ8M06vRbCoFrEZelP9kFhz4V6P72KBMV1GviTxFgeHxlCgH8b+UcWe6exL/rb57lEMsRYeEx5WzP8gFj84op0ML6vA+zCIf5T7Hatb0UnJ2t7EPQsgiVThKwnJRuqhOKt85SA/cZ2Jfr3We//5oWv8DrNdBuOB7+Vf+l6cX+L4zqdkk2njDqejSD/fFcvEOD/kqKqKWxMgc4VgvzkSG5i1lZt+TK4GdV8XQcksBTi5Fp5si1fGBoS11elX0+n/eul5mVhNqyDPJcD2ivT+yJoZabjGEh5eLX+98isYfxXb99vU4notYF/38i07mTyY7LEkSGCiV6/j360RACU07L1S71PQ7/Xx8Y/pjPUyxT4rhmCeCFHG5OMHDuBGcRCuXapgYWifmTq+Wx7goTrAVNFpqs/YUMpVk9whMVZouUccPE8aWhWy91ilb54jcyRWoIJDajoJeBS4X/wVQSwMEFAAAAAgAAAAhAEwp7OcyAAAAMAAAABoAAABzcmMvYWVyb3Zpc2lvbi9fX2luaXRfXy5weVNSUvLMS0ktSM1LSc0rUUhMLcpMzFFISS1JTS7JzM9TSK0oSC3KzE3NKynWU1JS4gIAUEsDBBQAAAAIAAAAIQAXjcG84AIAAEEJAAAVAAAAc3JjL2Flcm92aXNpb24vY2xpLnB5nVbBatwwEL37K4RO2rBxQhpKKfgQWhoCIVlKWyghGK093lUrS6o0dpJ+fZEse9NE66Q92SPPe/NmJM24sbolZdl02FkoSyJaoy0SrpRGjkIrl2Xjmt0Ybh2M9g+n1fiuXdZ4JsNxK8V6pFlx3GZZVkNDWi4UW7zPCCFEuxxUL6xWuQOsoeGdREa/X19elx+urz5dnJcfLz7TJXFomefIq7uaLcgRoblDjkD9ayfRcvmAonJ0sQjEQZ8lxaQ1P7ObrgWFq/CF1eAqK4xPrKBnYPU34YRW5JKviWu5lOR89ZUYYUAKBXRgdd2aFJE753Vdum49WM4TYkEr3bZc1XRJLPzqhIW6+GI7mOABNUAYrXWF2tIl2YI0BV1ZoZD0YL0QR7iqgwahGm3bsAVRhrFgSPGMzi9zO2r1VvjMY+KMHh5arZEuCT4YKHw5lyTWPFiM1hz5VMIUAVou1MggFO4I3h0fz+B6LpOok1mUA6iTsNOTAdUITBRiEDl5PKMNWf5VhcRmpYBgdLV1exLZDxPtxv1Oot6ezsDWHKttOth+kOItpE9fshLQi8r7j8y0Mt1M5V6zIc9Ad9r+BJsu20z6FlwXchl9r7SC6a5w3BLURHKHucEoGfrEaah0++haQP9/xyGBuwOx2aL7H+grTkQChULCP4N0D1ZyM+Iaqfnj+ucn++syczgS7lK0Auc2OSWuQ9PNdiTbKXc0bKFwvgPu5Wr5/WENaQVvxjaTwEHP5SzYn7sBze3GkYL03DoWp0B4eD7HorQ4AobZ43KjDZvGwuAhmp1TQcYxMEzEcBv8AM3h3oAVXuU4ROOo9EvZ5Gz81GB+BOd11xrHHnmxxZIIVYPyLSMWQD6JPk6Np+H9rRgDR5+nQeMy83ne0DBbbpch65vYgyfT9//JCF3kdp+gAflyNYLfTlIw2cGBDzEyO3iZJraHHVFcmKiyTDSkLH1jLcugsCz9P0xZRpHDD032B1BLAwQUAAAACAAAACEAmTFt7u0JAABYHAAAFgAAAHNyYy9hZXJvdmlzaW9uL2RhdGEucHmVWOuP47YR/+6/YsCiiHSnlb3bNLi65wIp7gIckOSCywNotwuDlkY27yRSJam1HcP/ezF8SPJjd1N92JXJ4bw58xsxxt6prawVL0FpsRaS1/CbMO+0kghcFxvxiAa4LKHhXxBKtKgbIYWxogDT8LoG060MWpNPJj+rThc4h421rZlPp2thN90qL1QzjTz7l5t33HKDdvKtFwKN0FrpOfxaW83rvRWFAW4cZ/hRQdcaq5E3UCj5iNoIJaFQJYIwgDssOotlPmGMTSaVVg0sl1VnO43LJYimVdoCl1JZboWSZjIJaxtuNrVYxZ+fjZLxXXNZqib+6nRdi1Wu8b8dGhtXfxdtJWr0EltuiVcU9xO3m17Onje1p/rpw/eR4kPD1ziZ/PjtD+9/hgXcTwAAWIslGqsFlyyLK6qtMf5aiWJfDD8LruPr43DE6q74Mvw4PcK3Usj1zfnyqjPxtVFWEduHyQ8fPn36+AkWwK4EtRtCNfWhmmqskRs00zKk1fRxls/yGZtMJiVWUIo1GpuQr+bORSnc/AOM1XMneQOLGJPcbPjdX79JUrexFXbjPJyrFmXC9IqlwA1U/hw9ldKwqlXxBYQEYVEnNW9WJZ9DlWvkZXI7u/saXgH9SzNYMZYOh53wvGtLbjFxXLxcjbbTEjb5BndB9zSY0nJtcKnV1iQWd3ZOVmSwFSWZJqTNYINivbHuhzPTdm2N97UwNgP6++DlM8Y+eTGPvBYlWK7XaP2tE2upNJagcU2JS/ztBlZqhwaKWrQtlmAV2A2CoHTK6QYQ08Ak6zks4P4hg/uHSfRVLSSSq0j53LS1sLRikpFXRAVSWUeZU0624016CiWtkB32i1ptSVJVK26Tx3godQIfSZrnRdISlrGURPRkD2PBNcpEq20Kb+HNqVDNhUH4jdcdvqeakVTs/a7FwmIJzuNDBasE1qWZw4HEHpmPKT27DPYZbDPYZGAKpTGDgltcK72HBVlxP/9mUGc3y2A/gwU0fJfM8lkGuzQbfuxHfG8z2N8SpZCJ94LLiDSDHbyGLR3rd3x+pBns4TVsBiaigt0tvF3AbgZKE7+3C9jPXnD9Su3I9V7XjDjceMXpZT87ca4zGRYLcAKEtEm0PqUgJbMMbm/PYh0SKedti7JMVmo3aIy1qMCpecrr7QJuz/QOiRnZHFikXoqSzc/O38BtBmy1Ujs2JwOPY5EGX8yLX+UXqbZyiO0hvsVkCBf8/LrEchWqWOIyNlxxrZQdFS968YrQRt58KYVOWq5RWrP4RXeYAe6EsUv1xf30cqlvSN4gLKBiMV/vZrd/u3n3/pebg5N3zH8Xrb/O1BNcYioL0/7wZHRJiSJ3gkySUlRpMTSoXJhleE2IbhTZlmti7E5TbVmarqrELmEkO6fd0a1pNYWn6jGDkGs4RGWOLIOq7sxmZCU9rmKd9s+807Wr4xU7+A5znJ7wsaJB1dnFNzNX5TWaVkmDmVM3tIDtZQtwngq6uZI3O9l6ukNEAX+sUThe+VaL017xhBavF66WPUEohiTDEv4MyV/uguiogbumlwqM4zH4bg6HEbvpFE6MOcIP4p9PhCmk0ZWMIZ+nL161D9J3r5H4gB/ncCAW4/LrwqixrXkRMnLo8lGDf4v2uz5hKdCRXc/lT/CxRcIyrv8VKAmOQCk0FpYuO6+NAo2fsbAGhCxU09ZosVfR5OfWBxE5+ZI69Hm/u9Z6mtbuR6aSumfFhZYiaNBIpSEZakgGVnMhl4XqpMcKsIA3s1lGaOB09Y5WDWIZF76+u1KBQpnINRpVP2IAUKqzbTcuIYxg/I09jKQfbx4PvdDjjTmQrKMvQKKCJPCYAmu4FBVdZELMLO3rzqiuhNR8TzsUIj8mzOHguTxVLYLLBlGlnxNygtBelSgcFnDozzHSlc2de7JhtagJmBo2BwezRzuuwtLG4ThajSPP+XqrlVWFqtkc2L8+fv+xB2mqETY0DQ/L/g74yOvODRphw0Ch1bYHcTTPWFBVJQoxHrcaJEhu8oDCjz1Sc6pm4MLi2nPCXNSoUA7RSzNI2COv2Shz0lFEgmlLQtFU8U9am+9q5O/eA0NMosPvh82He3fsgWLA6Law+YkAd4MyYB7Es3kE/mOa1Fvosouw67IUOoOar7CmV1iMssARGAbT6Ixhyx0Yti55/tGWTE8v/f85pLFQujTZeMTskcQy7PYAvAfhT1a8EyddrXy9fcTWKG2xTKTLFUkJclnG6P6yqT8yZUQj3Xwh81ptUSdpjrI0pE3C8s/tmqWnrSrUxxm8DWn41nc1z/FKf7xSKT/55o8lHHyFgQBzgikZ8Ecuar6qEQ4j5uPGQU+xUQblYLif1PNP7l9CBSDNDadaHziEq3NmEnnL54goM2iwWSFBYUDZNahpDvSCMjhHwmf4jcpv4s+nzuWXaMBXqz4jzyHcOa3HFsvV3qKJ2eChSRBziSNcIrkvCh4e9V3TCb0OH9xkEsfUqF9uxO+XWvUJlz+iFtU+dJXxw6VcBi8ugjv7Fj+kXgZsOromU5bmOoyDOSNX388e4DWw3O6sL/fjR/PtkuZVWMCJWwbZaV4ifRdKWGermzc3RqzP0ueJ6Xg0zkcp2amHLtn40uNu9mWCKB3EuCHbC7weh2EYhUWgvPdDzyXbQWycoK6SOA3YITD7ajRjffVwhEPihlGYwl0K0/DNIn9T0Y6bReNO+ILht7ZnpJtTgsto0XPpNHpGGfCiGUOPv/YwNzVStRjxTOE13GYvnAtXn2bOWAWePfHqlffmC3y5Rs7msIVXsHlJBeNwAZvD7GnKoUk+79ZkaJxTX5JiiUnPBju6WWkaagzl+XXfs/9Iln9WQnrOJs0AZaFo6Fv4y3UZ8Uu1Qvd7NspPR5j9wfgwsnUZcEi0+xlyl8YUJXe/n6bz6c3mIc+foTwHOq7+Xqe/jOiVsfAUPAxfS66lLTCHPGnJHztew24B8I6QW2De0xaqUCeg2lkmZKXO4LBf95Bs3sOf0+3RdSRsOEJGp3ShNAnH6v4QIp5BRJXy6Dt15qHN0J0doE8fBnaDX4dJpWLxUwoZF8eVUerTSl52TWsSoriS5OmzbONXsWc5n0XzWSFxcPrJT4plBEn0EZM+iQYWA2g6nBe/Y+w7Pisu5yw/NNB3docfD/o+pu+D87WDQVcSJ0wdD97RogIu98n4MDW6Me9nmdGo8jBCVheIkX0oUVpR8Nrb6j55orQehhvghVbGeIkOzLoPEM4JwZ+FkpVYn86JBKrZPMwPOTfLVhmxS0Z3NZg5jyk+DcPWQECqD9tu5ho2JW9cLpeisMl5ugYx3oNhEntu1nWz1CipaCc3vMIlZVbiDcwcEl5+wb1ZfMdrg08m2DMj/PXMjVQZCEnRWNw9yTtM7qTx5H9QSwMEFAAAAAgAAAAhABcHFKG2BgAAwxEAAB0AAABzcmMvYWVyb3Zpc2lvbi9kaWFnbm9zdGljcy5weZ1XS4/jNhK++1cUmMNIM7La6ezswbMaYDeLBAMEmcYmOQmGQUslm2mKFEj6hYb/e1AkZUl29+4mPrRaxWI9P35FMca+dmi4E2o777RQDtAYbcDgQeDxE9TCOqEqB43RLXz/9fuv8M8nwAOXe+6EVjljbDbzi+t1s3d7g+s1iLbTxgFXSjuvZmezKPvdahX0Ky0lVn6131Bjw/fS1aJyQafjbifFpl9/4m43m81qbEAf0EjeJTyDTQY1umBqzQ3y4gcuLabLGQDAKYNzBscMdlAADyKSkZCkJN54sVAOjQ12oICWn5JFBq1QyQk+kIWTfx5TmPvFE0nSFN6PVc/wAXZknp67XvVMkjT1bmpUuhWKO22ggCO8hx2I5iYFQGkxLpJP+mcH80mM3ppBtzdqGvvDxIe3Pbx+hkUwvsgXsZYtd9Vu7RtvE8fNFp3NoDNIjaD+ZCC2ShusM6i0akSNqsJikT9+zEDofbHIP8ZqM8Z+qbTBuTY1hg2SWzvnR24QtgaxPgd/Qm2h0QYOwu65hFrwrdLWicqCVvLsgUUW98qrYw0FWHSJ4WqLiUTVR5rGunpHohFes1x5GTkY8gChwGrjsE4myT3juZC83dQcuiV0JbOUAltldAyorMWvZt/jySOlGVm9qsM/RtUZlH1sWjmh9ngVVrrtuBMbiRSs8IEKim9IVzQQMyzFqmQVd7jV5rwWNVtBUUwimCyG1Om3QesikgeHk3zF8nqSxvY2G32i/McBBFma9ae0+FkrTMc18e6EBaUd0CJwVf9f5mnj4AE+F4SqaQWvdckNtvqACe0ZvNPvWShqPXNdQA79UIoGuDon/y2MjT7dUYjvOEWyyD/67mz0ifoTD8IIDGPXcXXs3+Lrqs0oygG6Oe86VHVCMY5izfy2gv70aNd75SwU8ELCJdh9m3Qlo5cAD+/H45/CHjykXuhXhYKSqpX5aLIh/NVl5KNkjSKTQGfu2oZ0zD5BMRt5yaAcsDOgO56+wcoqUlCYN4ndbyy6pSd64pq240ZYrQbJDfvgibedRFv8PXbEj42nLz/1I+NLy7eYhce/DT/OvJYze0fET9Mol5rXNom+4QHYgcu80pXOaZmluUFerx2eXBKLH+tE1TclE2Tbn7wlmJJt9AktC0kbSvptJ9HOa35CB2INoRiPxkSKHvrkg5MPn1DJRhOXrQbc9a3g42BXPdZ4sKV4ix5QVclCMlXJSBhzqUZ+It8ISjREapDK7fdf3bLOaKcrLdkS2I+B+y1vce5xAl/0b59pdAB30IgT1qP2fvIk0t83+oHBssH2oMuWo40jjVbXaNkSXi5BGALVe9ftiRUHdFE7/PSbBxiykWLePtfCJHgS1q31c+CFa+3JRThGx52WSGfICUknaCj+aNS82cfrrLrFyyTIhr2Qw0s+MvkadqaEM1Iuu1cB0A1BGKy0qe0wQfvI/LYRAvz7BGQDY0yJoBXW+rMyuWZMto0x6g0HBK4ml5Cblf7sbNElo5UMSppRAyAmjqYDIyY74VxvKoP37+9TKe6zKsIj0oJPQzsuAy/jOdCyKZ/xPOKD6DYSMZ7vebhRr7DxYL1kncFKWKEDMfdS17EVPPhx/202lX64vjYdW92GWzKDFZfyL1hTY2uRBMp49FYlPVcTTvAnM+xmy2gmm65GaC2hfHlewiGUKYODr10uHLY2Sem+8exJwldvaA1VLnSFrS53RV8NvoaqfgO/8BZ7iNcWeGW0tf5020/QCGPdddIAXWSrnbaoYHOOxHXgUtSed8FffIXa5pPjM46hXPa2bk5PCKAIwyrXHarRvAhVeThwyeCB5kwjJK4DP6d5pdUBjUvYf378F5uivDb82BulCZjTn4DzqaI/5g5bH+qkpDdx3n1W0a7+OnWnWWnpv3ReCDtLYFK0nigb/0YAH2F9CWxr+JldymAz3GfujVJSucHKcbWVmNC32DmD+J3mv73SjBhcCoWFjyCDo6jdrnhMXzfmufNuiX5knE7B44JMz+HbxzTNiIz9xAyBvhvdwN+tVhd4CWL/afButcwfmwvLoBFShnDuPN33IjBi7EYP6f/RibDnzV78qbKxM0qpj+yNyt3aWmSwyAKGc78hg8dFGnNmG8mr51eQ+Ubdk79l8F06JQZflzgCl/Drk/+cpLn4wxPQbeyhHzUhbni43tMIUp/8TChehslwGV0nrvZ9sMedcHizOg09pGn5AZN4ofDj2ZTvqP/LxXeXeZzVv3fbmPagGW4b19l9NMJhGN5+9tf7trNJJNMMhKpRueIxTpl4547Lsz8AUEsDBBQAAAAIAAAAIQAskhiloQsAAMkiAAAcAAAAc3JjL2Flcm92aXNpb24vZXhwZXJpbWVudC5wecUZa4/ctvH7/YopDQNaRyfvneO02UQBgthpAqSxkSfSxUHgSbO77EmkQlJ7u77efy+GpCRqH3bafKhwt5LImSE57xkxxn5seF3D39/+DFZzIYVcA5cVlKppuea3NcJXb756c2nsvkbALa87boWSGWPs4mKlVQNFsepsp7EoQDSt0ha4lMo6MHNxEcb8rRa3WYOWV9zyfuZfRsn+ua25XSnd9O+mu221KtGYfsSKBv26LbebWtz2i77ldjOstudNfXFxUeEKUG6FVrJBaZMZXH4BlSjt4gIAekyrdLm5cCNW7/0UXaVqGmEhj3aRlRss7wrV2bazyQBJ15KthWUpMI3by5Zrg/TyzesvX7GbFCzubP6T7jAFYyvUOo+Ivnr9y/c/f/fdQG6WGatFm8zcCO5KbC0kb358rbXSabydr3hdY/XWv7np2Yn9s07eSXUvmZvSaDst4WGAY+3ebpRki4H9mR8ptqiNUDKZpRFwgJmAh4cJoF+eLcI+YhK8vONrNGwR7YKudnFCT7J+F63nR3+tlIYWhISEdbXVvN5bURpiuhPo8LAVhE2vsmvaPT20+1KVyipVGzYSfYw331W84FsuarIBtvBKktFwJsw4Mznxuu2mkGu0RYVbUWIheYPJfAZidZ4UYG0QvlcSD3YyUA2scMge5jGoubNer5HEtIWzBw+CrSo3ZgFCBiGIZm3eRe+33Jab6J32ugBjtX/1J4gGDGLlwCGHj6/92L3Sd6j9KpDD3I9qNF2gBf92J4M8HNDZIm3yhC068ZKNR4LtIX57890bb6zeCiF3VBKmO2nYDJ677XuaqwCT4U4Ya5KZ821S2X5jA5s1Fwbha1HjawfqbClZsR86CbzWyKs9eCILePBEHz+DcqOUQeBS2Q1qtzAoDZeXnnzQrVLJlVhD7jzP6DRITLmxOnFKrtGoeovJLFIoL7jc38ZhJ77c/Y6DToa5+x0HveRyfxuHSX45/YxDQXx5uI8Tjaqwztle1erqSmatZeOcaq1oxDvUOfuy4s2v0VSt5/k8m8+voiVQrDdkDyXfu7n5y3FS3prDzfOm9R5zdGi83GD+Na9NNFihRd0IKYwV5QFCy61AWeIRC8taGSwaZbgo83l8Wj+SRVszJa8xn2cvxqFVLdpaT8Farf6Fpc29HkYn4w3m9BPJlRSpUHeHR2lrZc3BEQzfYtGiFqrKI2Y2fFdUaPMX87B7r2pPRkZCJQy5FWc4tSiFhbXmlUBpgZdl13S1C9ELsBsEjb93aCxWXpFAGMCdR8scZfLGkMdRIwqsbDENs5H7Ggi7QECGEE0ay7XFquhsSS5ONEiRb0UPCXv62+XT5vJp9dPTbxZP/7F4+uM/Weph1o2D6E3l0f3GTiBr7iqhk4HPxFLPoBZlRTnOkdNYsQcS0uMYdigvYTFSdq+FxYJCeUKTWdU1rUkIIQUhK5Q2v56lgLJUBJ+zzq4u/xa8wPsc/+iHIhCNBm3RIr8rGmyU3hfGcnJi3gcT6yD3DGlRr4pSddKiDvPObiF37jLx7ohcU2zIw76OnCGaribijkjmQ4sHCjnM1LEceo+Qt9TmQzSfPXu4W8DWhfK7FLYUzb2SZMJiQw5brOAO/pIDc3jscXbC9XvamTOVSujAALR8yQyWSlaG3ZxmFVx6Rv4XEvJ0nVhIQnqLVXG7txjWGAmQiQbB9YBBNknY/nNguCPbJqvx+jb7c0rWaiEneA+k4cwF4LDqLIWBK4tDLj3OUljVndlEJhPyRY8dsg1fHuDpfMM7ehMPjRkH5PDJx8FnWVHj0aDaoq55u4BVrbhLJbLrozyEktp5cLK1aIQ9yDv8Zv0OJrb+3O9cOF572OBKewqDR6Vip5hMnspgjqqJUrX7yfv2On51OShwA7J9T94TJagZPfVgVIudh6Idx5D0/uFcagDISJD9VCXWaGw0ScIaEFuNdPCLI0ZBPn1Vuudvb2L9zOdwNafpCfjngzQOMrNfeN2hT8rYD/h7JzQOlL7IiRRldhNaX+Q9BJv5jZoWS8hdcZgZvsKiVrzq0y9eeaM7MqxgBUoN/oboLBnVn+zGz1rdOUVzlkdUTZI4jOfAtrx20ukN/INLiYaKI3ImRHXJ/Du7WS6cqt8QF92TrxcOoDyJivAf9JKJit04/6rJuXogHzAP8CAPs/FkVMk7iCV3tDjROgkiVsADzcItTYtWxu/qidNKKLW6r2DDZVVTKBYGuByyDeBtq9VONI4kWAW/CPNKk92JtVQaK9C4ptV8XtIPnud9gPjD7Je4s4Ugig3fJQmPeHj23DMKiSve1Tafz+AjuPKGQ2xX944HfhOjWlPYVfcTVlFd4tk1gtFVKmmF7HxN09O9VTsCdjRu1Y4kOMUioJJbXCu9j7YdhsQxwjmxZ7ylBGja7eivaUlzeIkqD9xMz8MEBuQH7DiP0Z+KkPpnL6TzOLe3apffqt15CK6RE8Ty+gaeEXuXL95DTxinxXFKHl/TTsXpkV7PPsqDuqzJwZCFhDRhbZ1HNs6rUljJKsSWHhInpwGq1Mgtfisr3J3K/ij2h5gcVDxUxT5LpkAu+7pjrE3iBMD5/P8h+Q+rFmbDr19+whYhqgy7iUAbLsUKjT2C7U25Bwh2HLeTtLKqVDVbAIualV++TYOrmboUMowDV/KZsz21WolS8Hr0OKZUWsh1VM0xg9YKuabc6Vj3T1bmdFHwzOlnOhxynTzcp5OUCB/W0G4R1eXz7K/TwTOVflwqhvt0Mo6YefwygsWs7s2Tjm8U1W6JqKZyVJVv7IVumo81xHOa8e26+42qXXeUOFKxKLnW6t64Io9YnILL0ymSLW/S8D+APoFfuW6gawF5ufHUlaRQwkvbuUSIrzElXTeiwkDUBw23JaGNM6ztdSYaigrOUHp187Hw+ZbXDJ6HyLic3yzZStS+m8dugjU5oawCQWFcdjh1rWN36Xtlv1adrHwqc4Zs1FNwaVbizDn1K6TuqKnXNeJVTYcMCgReX4D0I+hEeqgAI/mSPHtVDPnGfBJhNJZKV2PWMD2SG/uj7POkzvEu8M9TPMm/9/HwFO1D0uc7rpQ1ej65IrOkDu5xVBzRzV6WG62keIfB1fbXhwryA5mS20lBQj4I+WjZIPWgx39S6hPq/xcGBavu04nkPeX4DJ7B1Xw+nxLw3qDHlweTE13+KIcaZRLxehbVA1PbVPcmw50lmk4OY0ISNMtlFvDsWTsbvznElA+ZSSv7vc7gKVy/hDyH+THTfK2+Yg8k2ccFPER4j8/dmz/N7DFYYEqnzB/CKR7Zca0+7S70tLNot+9vMxAzzjYXwvEIZnoaasF66pBTPkJJ+A8YqA2Q037QEV6U+BzPR4nQQ1+xLAauxEnt4lSim8IkpV3A8ubx3ErHyZTbvP/yqajz0JfWydqmEWYKjLJMdgIpo2+pjclEs/7WVWZR7DwP3fDdK/Th7yqFq3k6idY3JxDDU2x54+TQ9I2nn8BPGySjkxXXFZiuabjew4brqqRIDmET+dV87rT/y7ef9Qmh6vRQtkUEe4wX8zmYWpTUNZDGIq9ArbzWu6/M0AhTI3ft2Msr2CCnehDHCN1qLN23u9BUGM+4ZMMcu/G1vZfiChrkstjyWlQJjxILutwo5MCXnJzBfORg1OZyXafEgWZEK5m5HqQfMOIdjh/pxoU9Q5Yh+blZ0p3K5WO/zts82uFwiuUiBf83T+H6JkqoPNLL+Wm0+fvRCkNf9z+w4tUJVI0lr+tj9EMxeDhqTZwlxVv6fFGUNTcmn37upatcMh+0qRd5dpt3/Qn7VnFJThhl16AmhT5h81NXEn3bpYusQJb7on05LxqTe6HLNmuwElwmIVTF3+ImWJ++nGC1qEuUFJN7zBQ+fXmI7Y7numjxenzw+Qfgo2PJKRR4z3ycgvnZEChOzRfcFqP/LyiCTGJllOifCCH0FT/0Sz8QOZwJ/IEWNV1HbeqTBjQhdqYr7REv/gNQSwMEFAAAAAgAAAAhAEbuRGucAwAAwggAABcAAABzcmMvYWVyb3Zpc2lvbi90aWxlcy5weY1VTc+jNhC+51eMOJl9HUR2b2jZ2+6x6qHqBaHI4AlYBZvaJoG36n+vbPMRsm/VRkrAM49nnvlMFEXfp6ETtbBgRYfQoOrR6hmY5FB3zJgzezCNwEcHYxbBjMOg0RihZBJF0el006qH6/U22lHj9QqiH5S2wKRUllmhpDmdThxvYCzT1pAOZWPbDIS0FIx4x+VV3VF3bMjg1ilmYzh/g04YWwhpy+wEACBuEC7D1xxSUNpf3w5SWUjdabEEX+ESLrqPZsIg/M66Eb9rrTSJflVGWHFH4KJH6QIyPu5XG6Dxz1Fo5FHsrXXMWMihZxNJ6cro7LkEgLE4LIALBa1GyYln+gnIBc6r7TgOcI121BKM0hY5MWhJ8Ukz2aA375y9wYV6q3EQlHG85PQhJFcPQx6CbzltUTSt/f8JtuPQ4ZLihUuxpY1MFGYKvZBkgjdvjoL3FgfpvEmD3yUo97kpDTMIuVY+AAKnjc4RPj3BvZcP0eUS/aCRi9oSL+wVx46GRulZg3STZmCsXjWNeQ/58GfX9U/HY4qCrFbydhAINR7OHO+ifnbSs+nKMVQAcviSpvQULy0chsMqXbde4KfHH+/Cz5QazIqqmK1b5FfZm5MHtxQekIfwEtOyAYvscxlYancxh8I1TUrhQaGNSzcyLgOQ5xA9WtVhBNgZ3PuGOqMuCy8prtSEhoKplXZPvwvQOyjp8j1tRUspzCmF6UJhvrgKejZPs4dm7PzMuBolh7KtHx9UMafZfKEwpdl0KekLoDHvuf89KlyBcvfzgldjLtR4FIZa5eFxVC1Vy5fnUXlHXSmD+Q/Wmad7cZGGNPiUQb5EmvjsbQpxgwqEgV+URLeoOpSkil1N0uw1EivkiJtwmqcZcqgS95JwtKxuSZzUnZJI9slx2iKjUKQUPpclvOUwpT9rLxS+BO28az3VhA0DSk4cdDcbir/qqsTleCOxw5beeMJ1ZufaKdmsaHHzS9q7fO6OsHNKv0yJ753/aEI/MEnNLPGQmD5JAvogWi5uJLaRcI3Po53KH+g399PgkY85UNdecZEtzRKawE3WT7bWJOrGLXjC0dQouZBN/pse8dWEVo8wZNtwVWpavFNwf8CN0n6nvothHyBPsnAey6QeRhInVrndTuKV+b8ol3A+1Hrry97yfXQc89x53YuoHlsH/BWtRK+CR9lGm0JUVWqKMih2W3AOht3LnJYUIk84ygLxvw//kc7LoUv+AVBLAQIUABQAAAAIAAAAIQB4tUodYAEAAFECAAAOAAAAAAAAAAAAAACAAQAAAABweXByb2plY3QudG9tbFBLAQIUABQAAAAIAAAAIQCeDd5ZIQsAAK8XAAAJAAAAAAAAAAAAAACAAYwBAABSRUFETUUubWRQSwECFAAUAAAACAAAACEATCns5zIAAAAwAAAAGgAAAAAAAAAAAAAAgAHUDAAAc3JjL2Flcm92aXNpb24vX19pbml0X18ucHlQSwECFAAUAAAACAAAACEAF43BvOACAABBCQAAFQAAAAAAAAAAAAAAgAE+DQAAc3JjL2Flcm92aXNpb24vY2xpLnB5UEsBAhQAFAAAAAgAAAAhAJkxbe7tCQAAWBwAABYAAAAAAAAAAAAAAIABURAAAHNyYy9hZXJvdmlzaW9uL2RhdGEucHlQSwECFAAUAAAACAAAACEAFwcUobYGAADDEQAAHQAAAAAAAAAAAAAAgAFyGgAAc3JjL2Flcm92aXNpb24vZGlhZ25vc3RpY3MucHlQSwECFAAUAAAACAAAACEALJIYpaELAADJIgAAHAAAAAAAAAAAAAAAgAFjIQAAc3JjL2Flcm92aXNpb24vZXhwZXJpbWVudC5weVBLAQIUABQAAAAIAAAAIQBG7kRrnAMAAMIIAAAXAAAAAAAAAAAAAACAAT4tAABzcmMvYWVyb3Zpc2lvbi90aWxlcy5weVBLBQYAAAAACAAIABwCAAAPMQAAAAA=')
assert hashlib.sha256(bundle).hexdigest() == 'cbf2cd68c92de9631e4c3684845060d327784d064fcaee969d81dd73064a17fe'
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.namelist():
        assert (PROJECT / member).resolve().is_relative_to(PROJECT)
    archive.extractall(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
# Editable-install path hooks are not reloaded by an already-running kernel.
sys.path.insert(0, str(PROJECT / 'src'))
# Save the source that produced the run alongside its outputs.
(PERSIST / 'source-cbf2cd68c92d.zip').write_bytes(bundle)
def run(*args):
    command = [sys.executable, '-u', '-m', 'aerovision.cli', *map(str, args)]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
run('doctor')


In [ ]:
# Dataset stays on Colab scratch disk for fast image loading.
run('prepare', '--train', 800, '--val', 200)
DATA = PROJECT / 'data/mini-t800-v200-s42/dataset.yaml'
subset = DATA.parent
for filename in ['manifest.json', 'dataset.yaml']:
    (PERSIST / filename).write_bytes((subset / filename).read_bytes())
# Only small run artifacts/checkpoints go to Drive, not thousands of dataset images.
(PERSIST / 'runs').mkdir(exist_ok=True)
if not pathlib.Path('runs').exists():
    pathlib.Path('runs').symlink_to(PERSIST / 'runs', target_is_directory=True)
assert pathlib.Path('runs').resolve() == (PERSIST / 'runs').resolve()


In [ ]:
# Keep the same name to resume. Choose a NEW name for a separate experiment.
RUN_NAME = 'colab-pilot-v1'
EPOCHS = 20
run_dir = pathlib.Path('runs') / RUN_NAME
last = run_dir / 'weights/last.pt'
best = run_dir / 'weights/best.pt'
completed_epochs = 0
if (run_dir / 'results.csv').exists():
    with (run_dir / 'results.csv').open() as f:
        records = list(csv.DictReader(f))
    if records:
        completed_epochs = int(float(records[-1]['epoch']))
if completed_epochs >= EPOCHS and best.exists():
    print('Training already completed; proceeding to evaluation.')
elif last.exists():
    print('Resuming saved checkpoint; last recorded epoch:', completed_epochs)
    run('train', '--data', DATA, '--name', RUN_NAME, '--resume', last, '--device', '0')
else:
    if run_dir.exists():
        raise RuntimeError('An incomplete run folder has no checkpoint. Choose a new RUN_NAME.')
    run('train', '--data', DATA, '--epochs', EPOCHS, '--batch', 2,
        '--workers', 2, '--name', RUN_NAME, '--device', '0')


In [ ]:
# Both modes use the same checkpoint, images and prediction settings.
COMPARISON = pathlib.Path('runs') / (RUN_NAME + '-comparison')
run('compare', '--data', DATA, '--weights', best, '--output', COMPARISON, '--device', '0')
report = json.loads((COMPARISON / 'comparison.json').read_text())
for mode, metrics in report['modes'].items():
    print(mode, json.dumps(metrics, indent=2))


In [ ]:
# Compact result bundle, already stored in Drive; download a copy if desired.
export = PERSIST / (RUN_NAME + '-results.zip')
with zipfile.ZipFile(export, 'w', zipfile.ZIP_DEFLATED) as archive:
    for directory in [run_dir, COMPARISON]:
        for path in directory.rglob('*'):
            if path.is_file() and (path.suffix in {'.json', '.csv', '.yaml', '.png'}
                                  or path.name in {'best.pt', 'last.pt'}):
                # Exclude dataset preview images; results.png is a learning curve.
                if path.name.startswith(('train_batch', 'val_batch')) or path.name == 'labels.jpg':
                    continue
                archive.write(path, path.relative_to('runs'))
print('Saved:', export)
from google.colab import files
files.download(str(export))


## Interpretation
These are exploratory COCO-style validation metrics, not official VisDrone scores.
Ignored regions use an explicit crowd approximation. The validation set is fixed;
test-dev has not been used. Small-object recall gains must be weighed against latency
and false positives. Record the GPU actually assigned; do not compare its latency
directly with laptop timings.

Free GPU type and session duration vary: https://research.google.com/colaboratory/faq.html
After completing the experiment, disconnect the runtime to release its resources.
